## Market

In [1]:

%load_ext autoreload
%autoreload 1

%aimport quantdata.dysanos.grid.torch_market
%aimport quantdata.dysanos.grid.torch_pricer
%aimport quantdata.dysanos.grid.torch_model
%aimport quantdata.dysanos.grid.torch_gfx 
%aimport quantdata.dysanos.grid.torch_train 
%aimport quantdata.dysanos.grid.torch_simulator
%aimport quantdata.dysanos.grid.torch_universe
%aimport quantdata.dysanos.grid.torch_drift_change
%aimport quantdata.dysanos.grid.torch_det_measure_change
%aimport quantdata.dysanos.grid.torch_all

from quantdata.dysanos.grid.torch_all import run_experiment, DysanosExperiment
from cdxcore.subdir import SubDir
from cdxcore.uniquehash import named_unique_filename48_8
import gc
gc.collect()
import numpy as np

cache = SubDir("!/.noarbtest", fmt=SubDir.PYTREE_HDF5   )

experiment = DysanosExperiment(num_samples=40_000)

@cache.cache("0.1", uid = lambda experiment : named_unique_filename48_8(str(experiment), experiment))
def get_data(experiment):
    ex = run_experiment(
        experiment=experiment,
        plot_sampler=True,
        plot_det_measure_change=True)

    dC = ex.universe['C_tp1'] - ex.universe['C_t']
    dC = dC.reshape(dC.shape[0], dC.shape[1], -1)
    dS = ex.universe['spots'][:,1:] - ex.universe['spots'][:,:-1]
    dX = np.concatenate([dS[:,:,None], dC ], axis=2)
    X  = np.concatenate([ex.universe['spots'][:,:-1,None], ex.universe['C_t'].reshape(dC.shape[0], dC.shape[1], -1)], axis=2)
    return dict(X=X,dX=dX)

data = get_data(experiment)
dX = data['dX']
X = data['X']
print(get_data.cache_info.full_file_name)

print("*** READY ***")



/tmp/.noarbtest/DysanosExperiment(grid_strikes=100, num 33b74254.h5
*** READY ***


In [ ]:
import numpy as np
from quantdata.dysanos.grid.empirical_arbitrage import run_standard_experiments

results = run_standard_experiments(
    dC=dX,
    X=X,
    output_dir="empirical_arbitrage_results",
    threads=8
)


[empirical_arbitrage] starting 3 experiments; output directory: empirical_arbitrage_results
[empirical_arbitrage] experiment 1/3 delta0_prop0: delta=0, prop_cost=0
[empirical_arbitrage] preparing main LP: 40,000 paths, 1,269 positions
[empirical_arbitrage] starting HiGHS main solve with 8 threads
Running HiGHS 1.12.0 (git hash: 4f96ee8): Copyright (c) 2025 HiGHS under MIT licence terms
LP has 40000 rows; 2538 cols; 101519942 nonzeros
Coefficient ranges:
  Matrix  [1e-09, 2e-01]
  Cost    [3e-06, 1e-03]
  Bound   [1e+00, 1e+00]
  RHS     [1e-10, 1e-10]
Presolving model
40000 rows, 2538 cols, 101519942 nonzeros  16s
40000 rows, 1269 cols, 50759971 nonzeros  36s
Presolve reductions: rows 40000(-0); columns 1269(-1269); nonzeros 50759971(-50759971) 
Solving the presolved LP
Using EKK parallel dual simplex solver - PAMI with concurrency of 8
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 42s
        312    -9.3640851974e-02 Pr: 14429(483